In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/telco_churn.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['TotalCharges'].unique()[:20]

In [ ]:
df[df['TotalCharges'] == ' ']

In [ ]:
# Convert to numeric; anything that can't convert (like ' ') becomes NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many NaNs we now have
df['TotalCharges'].isnull().sum()

In [ ]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Confirm no more missing values
df['TotalCharges'].isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df['customerID'].duplicated().sum()

In [ ]:
df['Churn'].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.boxplot(y=df['tenure'], ax=axes[0])
axes[0].set_title('Tenure')

sns.boxplot(y=df['MonthlyCharges'], ax=axes[1])
axes[1].set_title('Monthly Charges')

sns.boxplot(y=df['TotalCharges'], ax=axes[2])
axes[2].set_title('Total Charges')

plt.tight_layout()
plt.show()

In [ ]:
def count_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    print(f"{column}: {len(outliers)} outliers (bounds: {lower:.2f} to {upper:.2f})")

for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    count_outliers_iqr(col)

In [ ]:
#cleaning done

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x='Churn', data=df)
plt.title('Customer Churn Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Contract', hue='Churn', data=df)
plt.title('Churn by Contract Type')
plt.xticks(rotation=15)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', bins=30)
plt.title('Churn by Tenure')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', multiple='stack', bins=30)
plt.title('Churn by Monthly Charges')
plt.show()

In [ ]:
# Convert Churn to 0/1 temporarily just for correlation purposes
df_corr = df.copy()
df_corr['Churn_numeric'] = df_corr['Churn'].map({'Yes': 1, 'No': 0})

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_numeric']

plt.figure(figsize=(7,5))
sns.heatmap(df_corr[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
#  EDA Key Insights

# - **Class imbalance**: ~74% of customers stayed, ~26% churned — accuracy alone will be a misleading metric later.
# - **Contract type is a major churn driver**: month-to-month customers churn far more than annual contract holders.
# - **Tenure matters**: churn is concentrated heavily among newer customers (low tenure).
# - **Pricing sensitivity**: customers with mid-to-high MonthlyCharges churn more, likely fiber/premium plan subscribers.
# - **Data quality**: 11 rows had blank TotalCharges (new customers with 0 tenure) — corrected to 0. No duplicates, no meaningful outliers.
# - **Feature redundancy risk**: tenure and TotalCharges are strongly correlated — worth simplifying in feature engineering.

In [ ]:
df['OnlineSecurity'].unique()

In [ ]:
service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

# Count how many of these equal exactly 'Yes' for each customer
df['ServiceCount'] = (df[service_cols] == 'Yes').sum(axis=1)

df['ServiceCount'].describe()

In [ ]:
df['ServiceCount'].value_counts().sort_index()

In [ ]:
rfm = df[['customerID', 'tenure', 'ServiceCount', 'MonthlyCharges']].copy()
rfm.columns = ['customerID', 'Recency', 'Frequency', 'Monetary']
rfm.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

In [ ]:
from sklearn.cluster import KMeans

inertia = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(7,4))
plt.plot(K_range, inertia, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    score = silhouette_score(rfm_scaled, labels)
    print(f"K={k}: Silhouette Score = {score:.3f}")

In [ ]:
optimal_k = 4  # replace with what your elbow + silhouette actually show

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Cluster', palette='Set2')
plt.title('Customer Segments (Frequency vs Monetary)')
plt.show()

In [ ]:
plt.savefig('../images/customer_segments.png', dpi=150, bbox_inches='tight')

In [ ]:
df_model = df.drop(columns=['customerID'])

# Merge our RFM cluster back in as a feature (optional but strong — ties both halves of project together)
df_model = df_model.merge(rfm[['customerID', 'Cluster']], left_index=True, right_index=False, how='left') if False else df_model
# (simpler: just re-attach using index alignment since row order matches)
df_model['Cluster'] = rfm['Cluster'].values

# Encode target
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})

# One-hot encode categorical columns
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

df_model.shape

In [ ]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)

log_reg.fit(X_train, y_train)
rf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                               f1_score, confusion_matrix, roc_auc_score, classification_report)

def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"--- {name} ---")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
    print(f"Precision: {precision_score(y_test, y_pred):.3f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.3f}")
    print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.3f}")
    print()
    
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
    plt.title(f'Confusion Matrix - {name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

evaluate_model(log_reg, X_test, y_test, "Logistic Regression")
evaluate_model(rf, X_test, y_test, "Random Forest")

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,6))
importances.head(15).plot(kind='barh')
plt.title('Top 15 Feature Importances - Random Forest')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
import joblib

joblib.dump(rf, '../models/churn_rf_model.pkl')
joblib.dump(log_reg, '../models/churn_logreg_model.pkl')
joblib.dump(scaler, '../models/rfm_scaler.pkl')
joblib.dump(kmeans, '../models/kmeans_model.pkl')
joblib.dump(X.columns.tolist(), '../models/model_columns.pkl')  # needed to align dashboard input columns later

In [ ]:
# ## Business Insights

# **Most valuable customers:** Long-tenure customers with high service adoption and moderate-to-high monthly charges (Cluster: Loyal High-Value). Retention here should focus on loyalty rewards, not discounts — they're already committed.

# **Highest churn risk:** New customers (low tenure) on month-to-month contracts with high monthly charges — especially those with low service adoption (haven't found reasons to stay engaged).

# **Retention strategies by segment:**
# - New/At-Risk customers → onboarding campaigns, first-90-days engagement nudges, highlight underused features/services
# - High-Value New customers → priority personal outreach, contract-upgrade incentives (month-to-month → annual, often with a discount tied to commitment)
# - Stable Low-Spend customers → upsell opportunities, bundle offers

# **Revenue improvement:** Converting month-to-month customers to annual contracts is the single highest-leverage action — the data shows contract type is the strongest churn driver. Even a modest incentive (small discount) may be cheaper than repeated re-acquisition costs.

In [ ]:
# import streamlit as st
# import pandas as pd
# import joblib

# st.set_page_config(page_title="Customer Churn & Segmentation", layout="wide")

# st.title("📊 Customer Segmentation & Churn Prediction Dashboard")

# # Load saved models
# rf_model = joblib.load('../models/churn_rf_model.pkl')
# scaler = joblib.load('../models/rfm_scaler.pkl')
# kmeans = joblib.load('../models/kmeans_model.pkl')
# model_columns = joblib.load('../models/model_columns.pkl')

# uploaded_file = st.file_uploader("Upload customer CSV", type=['csv'])

# if uploaded_file:
#     data = pd.read_csv(uploaded_file)
#     st.subheader("Uploaded Data Preview")
#     st.dataframe(data.head())

#     # --- Segmentation ---
#     st.subheader("Customer Segments")
#     service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
#                      'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
#     data['ServiceCount'] = (data[service_cols] == 'Yes').sum(axis=1)
    
#     rfm_input = data[['tenure', 'ServiceCount', 'MonthlyCharges']]
#     rfm_scaled = scaler.transform(rfm_input)
#     data['Cluster'] = kmeans.predict(rfm_scaled)
    
#     st.bar_chart(data['Cluster'].value_counts())
#     st.dataframe(data[['customerID', 'tenure', 'ServiceCount', 'MonthlyCharges', 'Cluster']])

#     # --- Churn Prediction ---
#     st.subheader("Churn Predictions")
    
#     df_pred = data.drop(columns=['customerID']) if 'customerID' in data.columns else data.copy()
#     if 'TotalCharges' in df_pred.columns:
#         df_pred['TotalCharges'] = pd.to_numeric(df_pred['TotalCharges'], errors='coerce').fillna(0)
    
#     cat_cols = df_pred.select_dtypes(include='object').columns.tolist()
#     df_encoded = pd.get_dummies(df_pred, columns=cat_cols, drop_first=True)
    
#     # Align columns to match training data exactly
#     df_encoded = df_encoded.reindex(columns=model_columns, fill_value=0)
    
#     predictions = rf_model.predict(df_encoded)
#     probabilities = rf_model.predict_proba(df_encoded)[:, 1]
    
#     data['Churn_Prediction'] = ['Yes' if p == 1 else 'No' for p in predictions]
#     data['Churn_Probability'] = probabilities.round(3)
    
#     st.dataframe(data[['customerID', 'Churn_Prediction', 'Churn_Probability']])
    
#     st.subheader("Retention Recommendations")
#     high_risk = data[data['Churn_Probability'] > 0.5]
#     st.write(f"⚠️ {len(high_risk)} customers flagged as high churn risk.")
#     st.dataframe(high_risk[['customerID', 'Churn_Probability', 'Cluster']])
# else:
#     st.info("Upload a CSV file to get started.")

In [117]:
import os

# Define the code content for your Streamlit app
app_code = """import os
import streamlit as st
import pandas as pd
import joblib

# Force Python to use your project root directory as the base path so it finds /models
os.chdir(r"g:\\projects\\churn")

st.set_page_config(page_title="Customer Churn & Segmentation", layout="wide")
st.title("📊 Customer Segmentation & Churn Prediction Dashboard")

# Load saved models
rf_model = joblib.load('models/churn_rf_model.pkl')
scaler = joblib.load('models/rfm_scaler.pkl')
kmeans = joblib.load('models/kmeans_model.pkl')
model_columns = joblib.load('models/model_columns.pkl')

uploaded_file = st.file_uploader("Upload customer CSV", type=['csv'])

if uploaded_file:
    data = pd.read_csv(uploaded_file)
    st.subheader("Uploaded Data Preview")
    st.dataframe(data.head())

    # --- Segmentation ---
    st.subheader("Customer Segments")
    service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    data['ServiceCount'] = (data[service_cols] == 'Yes').sum(axis=1)
    
    rfm_input = data[['tenure', 'ServiceCount', 'MonthlyCharges']]
    rfm_scaled = scaler.transform(rfm_input)
    data['Cluster'] = kmeans.predict(rfm_scaled)
    
    st.bar_chart(data['Cluster'].value_counts())
    st.dataframe(data[['customerID', 'tenure', 'ServiceCount', 'MonthlyCharges', 'Cluster']])

    # --- Churn Prediction ---
    st.subheader("Churn Predictions")
    
    df_pred = data.drop(columns=['customerID']) if 'customerID' in data.columns else data.copy()
    if 'TotalCharges' in df_pred.columns:
        df_pred['TotalCharges'] = pd.to_numeric(df_pred['TotalCharges'], errors='coerce').fillna(0)
    
    cat_cols = df_pred.select_dtypes(include='object').columns.tolist()
    df_encoded = pd.get_dummies(df_pred, columns=cat_cols, drop_first=True)
    
    # Align columns to match training data exactly
    df_encoded = df_encoded.reindex(columns=model_columns, fill_value=0)
    
    predictions = rf_model.predict(df_encoded)
    probabilities = rf_model.predict_proba(df_encoded)[:, 1]
    
    data['Churn_Prediction'] = ['Yes' if p == 1 else 'No' for p in predictions]
    data['Churn_Probability'] = probabilities.round(3)
    
    st.dataframe(data[['customerID', 'Churn_Prediction', 'Churn_Probability']])
    
    st.subheader("Retention Recommendations")
    high_risk = data[data['Churn_Probability'] > 0.5]
    st.write(f"⚠️ {len(high_risk)} customers flagged as high churn risk.")
    st.dataframe(high_risk[['customerID', 'Churn_Probability', 'Cluster']])
else:
    st.info("Upload a CSV file to get started.")
"""

# '../app/app.py' goes up one level out of 'notebooks' and enters your root 'app' folder
target_path = "../app/app.py"

with open(target_path, "w", encoding="utf-8") as f:
    f.write(app_code)

print("Success! File successfully written to your root folder: churn/app/app.py")

Success! File successfully written to your root folder: churn/app/app.py
